# PCA: geometría, cálculo e interpretación

Este notebook acompaña el capítulo de análisis de componentes principales. Reproduce una demostración bidimensional y una adaptación docente de los datos de cuatro pozos geotérmicos publicados por McKillup y Dyar (2010, tabla 20.7). El libro usa esos datos para otros métodos multivariados; el PCA es una adaptación de la materia.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Geometría en dos dimensiones

Los puntos permanecen iguales; PCA cambia el sistema de coordenadas. Los autovectores de la matriz de covarianzas definen los ejes y los autovalores indican la varianza proyectada.

In [ ]:
rng = np.random.default_rng(20261016)
R = np.array([[1.0, 0.859], [0.859, 1.0]])
Z = rng.multivariate_normal([0, 0], R, size=150)
S = np.cov(Z, rowvar=False, ddof=1)
lam, A = np.linalg.eigh(S)
orden = np.argsort(lam)[::-1]
lam, A = lam[orden], A[:, orden]
T = Z @ A

fig, ax = plt.subplots(1, 2, figsize=(10, 4.3), constrained_layout=True)
ax[0].scatter(Z[:, 0], Z[:, 1], s=22, alpha=.55, color='#446681')
for k, color in enumerate(['#B56F3A', '#76557C']):
    v = A[:, k] * 2.2 * np.sqrt(lam[k])
    ax[0].plot([-v[0], v[0]], [-v[1], v[1]], lw=2.5, color=color, label=f'PC{k+1}')
ax[0].set(xlabel='Variable 1 (z)', ylabel='Variable 2 (z)', title='Ejes originales')
ax[0].legend()
ax[1].scatter(T[:, 0], T[:, 1], s=22, alpha=.55, color='#446681')
ax[1].axhline(0, color='#76557C'); ax[1].axvline(0, color='#B56F3A')
ax[1].set(xlabel='Score PC1', ylabel='Score PC2', title='Nuevas coordenadas')
lim = 1.08*np.max(np.abs(np.r_[Z.ravel(), T.ravel()]))
for a in ax:
    a.set(xlim=(-lim, lim), ylim=(-lim, lim))
    a.set_aspect('equal', adjustable='box')
plt.show()
pd.DataFrame({'autovalor': lam, 'proporción': lam/lam.sum()}, index=['PC1', 'PC2'])

## 2. Cuatro pozos y seis elementos

Con cuatro observaciones centradas, el rango no puede superar tres. Por eso habrá como máximo tres autovalores positivos, aunque tengamos seis variables. Estandarizamos cada elemento antes de diagonalizar la matriz de correlaciones.

In [ ]:
geo = pd.DataFrame({
    'Ce': [12, 16, 22, 14], 'Nd': [43, 54, 6, 39],
    'Eu': [32, 34, 54, 28], 'Tb': [61, 23, 32, 71],
    'Ho': [2, 7, 10, 8], 'Tm': [31, 65, 4, 29],
}, index=['A', 'B', 'C', 'D'])
Zg = (geo-geo.mean())/geo.std(ddof=1)
Rg = Zg.cov().to_numpy()
lam, A = np.linalg.eigh(Rg)
orden = np.argsort(lam)[::-1]
lam, A = lam[orden], A[:, orden]
Tg = Zg.to_numpy() @ A
L = A * np.sqrt(np.clip(lam, 0, None))
explicada = lam/lam.sum()
pd.DataFrame({'autovalor': lam, 'proporción': explicada, 'acumulada': np.cumsum(explicada)},
             index=[f'PC{i}' for i in range(1, 7)])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
ax[0].plot(range(1, 7), lam, 'o-', color='#B56F3A')
ax[0].axhline(1, color='#7B8794', ls='--', lw=1)
ax[0].set(xlabel='Componente', ylabel='Autovalor', title='Scree plot')
ax[1].scatter(Tg[:, 0], Tg[:, 1], s=80, color='#446681')
for i, pozo in enumerate(geo.index):
    ax[1].annotate(pozo, Tg[i, :2], xytext=(5, 5), textcoords='offset points')
ax[1].set(xlabel=f'PC1 ({explicada[0]:.1%})', ylabel=f'PC2 ({explicada[1]:.1%})', title='Scores')
ax[2].add_patch(plt.Circle((0, 0), 1, fill=False, color='#AAB4BD'))
for j, nombre in enumerate(geo.columns):
    ax[2].arrow(0, 0, L[j, 0], L[j, 1], color='#76557C', head_width=.035, length_includes_head=True)
    ax[2].text(1.08*L[j, 0], 1.08*L[j, 1], nombre, ha='center', va='center')
ax[2].set(xlim=(-1.15, 1.15), ylim=(-1.15, 1.15), aspect='equal',
          xlabel='Correlación con PC1', ylabel='Correlación con PC2', title='Variables')
plt.show()

## 3. Comprobaciones e interpretación

- La suma de autovalores debe coincidir con la traza de la matriz de correlaciones: seis.
- Los scores de componentes distintas deben tener covarianza aproximadamente cero.
- Cambiar el signo de una columna de `A` y de la columna correspondiente de `Tg` no cambia el análisis.
- La proximidad de A y D describe semejanza química en estas variables; no demuestra conectividad hidráulica.
- Con sólo cuatro pozos no puede evaluarse estabilidad ni generalizarse a una población.

In [ ]:
print('Suma de autovalores:', lam.sum())
print('Traza de R:', np.trace(Rg))
print('Covarianza de scores (redondeada):')
print(np.round(np.cov(Tg, rowvar=False, ddof=1), 6))